# 04 — Lambda Inference Handler Demo

This notebook accompanies `04-serverless-model-deployment-lambda-api-gateway.md`.

It writes an actual `lambda_handler(event, context)` function — the exact entry-point shape AWS
Lambda invokes — implementing the request-parsing, mock-inference, and response-formatting logic a
real chart-detection Lambda would have behind API Gateway. A stub "model" (returning fixed synthetic
detections) stands in for the real fine-tuned YOLOv5 model, so this runs completely offline: no
AWS credentials, no deployed Lambda, no real model weights required.

We then call the handler function directly with a fake API-Gateway-shaped `event` dict to show it
working end to end, including its error-handling path for malformed requests.

## 1. A stub model standing in for the fine-tuned YOLOv5 model

In production, this class would wrap something like `torch.hub.load(...)` loading the fine-tuned
`weights.pt` artifact from Chapter 02, or a TorchScript/ONNX Runtime session for faster cold starts
(Chapter 04). Here it just returns two fixed, synthetic detections, so the rest of the handler logic
can be exercised without any real model or GPU.

In [1]:
import base64
import io
import json
import time

from PIL import Image
import numpy as np


class StubChartDetectorModel:
    """
    Stand-in for a loaded YOLOv5 model. In production this would wrap
    torch.hub.load(...) or a TorchScript/ONNX runtime session and run a real
    forward pass. Here it returns fixed, synthetic detections so the handler
    logic can be demonstrated fully offline with no model weights or AWS
    credentials required.
    """

    def __init__(self):
        self.loaded_at = time.time()

    def predict(self, image: Image.Image, confidence_threshold: float = 0.25):
        width, height = image.size
        synthetic_detections = [
            {"box": [0.10, 0.15, 0.45, 0.55], "confidence": 0.96, "class": "chart"},
            {"box": [0.55, 0.20, 0.90, 0.50], "confidence": 0.31, "class": "chart"},
        ]
        results = []
        for det in synthetic_detections:
            if det["confidence"] < confidence_threshold:
                continue
            x1, y1, x2, y2 = det["box"]
            results.append({
                "class": det["class"],
                "confidence": round(det["confidence"], 4),
                "bbox_pixels": [
                    round(x1 * width), round(y1 * height),
                    round(x2 * width), round(y2 * height),
                ],
                "bbox_normalized": det["box"],
            })
        return results


# Loaded once at MODULE scope, not inside the handler -- this is the cold-start
# mitigation described in Chapter 04: the (simulated) expensive model-load cost
# is paid once per warm Lambda container, and every subsequent invocation of
# that same container reuses the already-loaded model.
_MODEL = StubChartDetectorModel()
print("Model 'loaded' at:", _MODEL.loaded_at)

Model 'loaded' at: 1784106446.006361


## 2. Request parsing

API Gateway (in Lambda proxy integration mode) hands the Lambda function an `event` dict where the
actual request body is a JSON *string* under `event['body']`, optionally base64-encoded for binary
payloads. We accept a JSON body containing a base64-encoded image (`image_base64`), decode it into a
PIL image, and raise a clear error for anything malformed -- exactly the kind of defensive parsing a
production handler needs, since it's the public internet-facing boundary of the system.

In [2]:
def _decode_image_from_event(event: dict) -> Image.Image:
    body = event.get("body")
    if body is None:
        raise ValueError("Request body is missing")

    if event.get("isBase64Encoded"):
        raw_bytes = base64.b64decode(body)
    else:
        payload = json.loads(body) if isinstance(body, str) else body
        image_b64 = payload.get("image_base64")
        if not image_b64:
            raise ValueError("Missing 'image_base64' field in request body")
        raw_bytes = base64.b64decode(image_b64)

    return Image.open(io.BytesIO(raw_bytes)).convert("RGB")


print("_decode_image_from_event defined.")

_decode_image_from_event defined.


## 3. The Lambda handler itself

This mirrors the real shape of a chart-detection Lambda sitting behind API Gateway: parse the
request, run inference via the (stub) model, and return an API-Gateway-proxy-shaped response dict
with `statusCode`, `headers`, and a JSON-string `body`. Malformed requests get a structured `400`
response rather than an unhandled exception -- unhandled exceptions in a real Lambda show up to the
caller as an opaque `502` from API Gateway, which is much harder to debug from the client side.

In [3]:
def lambda_handler(event, context):
    """
    Mirrors the shape of a real chart-detection Lambda handler sitting behind
    API Gateway: parse the incoming request, run inference, and format a
    JSON HTTP response -- with structured error handling for bad input.
    """
    try:
        image = _decode_image_from_event(event)
    except Exception as exc:
        return {
            "statusCode": 400,
            "headers": {"Content-Type": "application/json"},
            "body": json.dumps({"error": f"Invalid request: {exc}"}),
        }

    query_params = event.get("queryStringParameters") or {}
    confidence_threshold = float(query_params.get("confidence_threshold", 0.25))

    start = time.time()
    detections = _MODEL.predict(image, confidence_threshold=confidence_threshold)
    inference_ms = round((time.time() - start) * 1000, 2)

    response_body = {
        "image_size": list(image.size),
        "num_detections": len(detections),
        "detections": detections,
        "inference_ms": inference_ms,
        "model_loaded_at": _MODEL.loaded_at,
    }

    return {
        "statusCode": 200,
        "headers": {"Content-Type": "application/json"},
        "body": json.dumps(response_body),
    }


print("lambda_handler defined.")

lambda_handler defined.


## 4. Building a fake API Gateway event and calling the handler

We synthesize a small random RGB image with PIL/numpy, base64-encode it into a JSON body exactly the
way API Gateway's Lambda proxy integration would deliver a real request, and call `lambda_handler`
directly -- no deployed infrastructure required.

In [4]:
synthetic_image = Image.fromarray(
    (np.random.default_rng(0).random((300, 400, 3)) * 255).astype(np.uint8)
)
buf = io.BytesIO()
synthetic_image.save(buf, format="PNG")
image_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")

fake_event = {
    "httpMethod": "POST",
    "path": "/detect-charts",
    "isBase64Encoded": False,
    "queryStringParameters": {"confidence_threshold": "0.5"},
    "body": json.dumps({"image_base64": image_b64}),
}

fake_context = {}  # Lambda's real context object is unused by this handler

result = lambda_handler(fake_event, fake_context)
print("Status code:", result["statusCode"])
print(json.dumps(json.loads(result["body"]), indent=2))

assert result["statusCode"] == 200
parsed = json.loads(result["body"])
# Only the 0.96-confidence synthetic detection clears the 0.5 confidence_threshold we passed in
assert parsed["num_detections"] == 1
print("\nOK -- handler returned exactly the one detection above the requested confidence threshold.")

Status code: 200
{
  "image_size": [
    400,
    300
  ],
  "num_detections": 1,
  "detections": [
    {
      "class": "chart",
      "confidence": 0.96,
      "bbox_pixels": [
        40,
        45,
        180,
        165
      ],
      "bbox_normalized": [
        0.1,
        0.15,
        0.45,
        0.55
      ]
    }
  ],
  "inference_ms": 0.0,
  "model_loaded_at": 1784106446.006361
}

OK -- handler returned exactly the one detection above the requested confidence threshold.


## 5. Exercising the error path

A request with no `image_base64` field should come back as a clean `400`, not an unhandled
exception -- this is exactly the defensive-parsing behavior worth pointing to in an interview when
asked "how does your Lambda handle bad input."

In [5]:
bad_event = {"body": json.dumps({})}
bad_result = lambda_handler(bad_event, fake_context)

print("Bad request status code:", bad_result["statusCode"])
print(bad_result["body"])

assert bad_result["statusCode"] == 400
print("\nOK -- malformed request handled gracefully with a structured 400 response.")

Bad request status code: 400
{"error": "Invalid request: Missing 'image_base64' field in request body"}

OK -- malformed request handled gracefully with a structured 400 response.


## Tying it back

This is the exact function packaged into the container-image Lambda deployment described in Chapter
04: `_MODEL` loaded once at module scope (the cold-start mitigation), a defensive request-parsing
layer that turns malformed input into a clean `400` instead of a crash, and a JSON response shape
that API Gateway passes straight through to the caller. Swap `StubChartDetectorModel` for a real
`torch.hub.load(...)`-based wrapper around the fine-tuned YOLOv5 weights, and this handler is
production-shaped as written.